# Python Refresh for ML

This notebook does not reteach Python from scratch. It reviews only the Python pieces you will use frequently in later `PyTorch` work.

Focus:

- Function arguments
- Classes, inheritance, and `super()`
- Special methods `__len__` and `__getitem__`
- Context managers, assertions, and exceptions

How to use this notebook:

1. Read first and predict the output.
2. Then run the example code.
3. Complete the `TODO` exercises yourself.
4. Check the reference solutions last.

## Learning Goals

After finishing this notebook, you should be able to:

1. Read function signatures used in training scripts.
2. Understand the Python foundations behind `nn.Module` and `Dataset`.
3. Write simple classes that implement `__len__` and `__getitem__`.
4. Use `assert` and exceptions for basic input validation.
5. Map these Python concepts to later `PyTorch` usage.

## Function Arguments

Function argument design appears everywhere in machine learning code.

Common cases:

- Passing hyperparameters
- Providing default values
- Improving readability with keyword-only arguments
- Accepting flexible configuration values

In [ ]:
def build_experiment_name(model_name, lr=1e-3, batch_size=32, *, seed=42, extra_tags=None):
    if extra_tags is None:
        extra_tags = []

    tag_text = "-".join(extra_tags) if extra_tags else "base"
    return f"{model_name}_lr{lr}_bs{batch_size}_seed{seed}_{tag_text}"


name = build_experiment_name(
    "mlp",
    batch_size=64,
    seed=7,
    extra_tags=["aug", "dropout"],
)

print("experiment name / experiment name:", name)

# Think:
# Why must seed be passed as seed=7 after the * marker?


Key ideas:

- Default arguments: `lr=1e-3`, `batch_size=32` supply values when the caller leaves them out.
- The bare `*` in a function signature is a separator. Parameters after it are keyword-only, so `seed` and `extra_tags` must be written as `seed=...` and `extra_tags=...`.
- Keyword-only arguments make training code harder to misread. `build_experiment_name("mlp", 0.01, 128, 7)` is ambiguous, while `seed=7` is explicit.
- Mutable default pitfall: `extra_tags=None` is safer than `extra_tags=[]`, because a default list would be shared across calls.

This pattern is very common in training utilities.

In [ ]:
# Exercise 1
#
# Implement summarize_split().
#
# Inputs:
# - train_size
# - val_size
# - test_size
# - shuffle
# - stratify
#
# Requirements:
# - test_size defaults to 0
# - shuffle and stratify must be keyword-only
# - return a dict containing train_size, val_size, test_size, total_size, shuffle, and stratify

def summarize_split(train_size, val_size, test_size=0, *, shuffle=True, stratify=False):
    # write your implementation here
    pass


# print(summarize_split(800, 100, test_size=100, shuffle=True, stratify=True))

In [ ]:
# Exercise 1 Reference Solution

def summarize_split_solution(train_size, val_size, test_size=0, *, shuffle=True, stratify=False):
    total_size = train_size + val_size + test_size
    return {
        "train_size": train_size,
        "val_size": val_size,
        "test_size": test_size,
        "total_size": total_size,
        "shuffle": shuffle,
        "stratify": stratify,
    }


print(summarize_split_solution(800, 100, test_size=100, shuffle=True, stratify=True))

### `*args` and `**kwargs`

These names describe two different ways to collect extra arguments:

- `*args` collects extra positional arguments into a tuple.
- `**kwargs` collects extra keyword arguments into a dict.

Example:

```python
log_metrics(3, 0.91, 0.27, loss=0.27, accuracy=0.91)
```

Inside `log_metrics`:

- `epoch == 3`
- `metric_values == (0.91, 0.27)`
- `named_metrics == {"loss": 0.27, "accuracy": 0.91}`

In ML code, `*args` is common when a wrapper forwards unnamed values, and `**kwargs` is common when passing flexible config options such as `batch_size=32` or `num_workers=2`.

Use them when the function truly needs flexibility. For beginner project code, explicit parameters are usually clearer.

In [ ]:
def log_metrics(epoch, *metric_values, **named_metrics):
    print(f"epoch={epoch}")
    print("positional metrics / positional metrics:", metric_values)
    print("named metrics / named metrics:", named_metrics)


log_metrics(3, 0.91, 0.27, loss=0.27, accuracy=0.91)

## Classes, Inheritance, and `super()`

`PyTorch` model and dataset code relies heavily on object-oriented programming.

You should at least be comfortable with:

- initializing objects with `__init__`
- instance attributes
- child classes inheriting from parent classes
- using `super()` to call parent logic

In [ ]:
class MetricTracker:
    def __init__(self, name):
        self.name = name
        self.values = []

    def update(self, value):
        self.values.append(float(value))

    def compute(self):
        if not self.values:
            return 0.0
        return sum(self.values) / len(self.values)


class RunningAverage(MetricTracker):
    def __init__(self, name):
        super().__init__(name)
        self.total = 0.0
        self.count = 0

    def update(self, value):
        value = float(value)
        self.total += value
        self.count += 1
        self.values.append(value)

    def compute(self):
        if self.count == 0:
            return 0.0
        return self.total / self.count


loss_tracker = RunningAverage("train_loss")
for loss in [0.95, 0.72, 0.51]:
    loss_tracker.update(loss)

print("name / name:", loss_tracker.name)
print("average / average:", loss_tracker.compute())

You should understand this mapping:

- stores shared logic
- extends or overrides behavior
- first run the parent initialization

Writing `nn.Module` subclasses later follows almost the same pattern.


In [ ]:
# Exercise 2
# 
# Goal:
# number of batches
# total number of samples
# average batch size

class BatchCounter:
    def __init__(self):
        # TODO
        pass

    def update(self, batch):
        # TODO
        pass

    def summary(self):
        # TODO
        pass


# counter = BatchCounter()
# counter.update([1, 2, 3, 4])
# counter.update([5, 6])
# print(counter.summary())

In [ ]:
# Exercise 2 Reference Solution

class BatchCounterSolution:
    def __init__(self):
        self.num_batches = 0
        self.num_samples = 0

    def update(self, batch):
        self.num_batches += 1
        self.num_samples += len(batch)

    def summary(self):
        avg_batch_size = 0.0 if self.num_batches == 0 else self.num_samples / self.num_batches
        return {
            "num_batches": self.num_batches,
            "num_samples": self.num_samples,
            "avg_batch_size": avg_batch_size,
        }


counter = BatchCounterSolution()
counter.update([1, 2, 3, 4])
counter.update([5, 6])
print(counter.summary())

## `__len__` and `__getitem__`

This is one of the most important parts of this notebook because it maps directly to the minimal `PyTorch Dataset` interface.

If a class implements:

- `__len__`
- `__getitem__`

it can behave like an indexable data container.


In [ ]:
class ToyDataset:
    def __init__(self, samples):
        self.samples = list(samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        features, label = self.samples[index]
        return {"x": features, "y": label}


samples = [
    ([0.1, 0.2], 0),
    ([0.7, 0.9], 1),
    ([0.3, 0.4], 0),
]

dataset = ToyDataset(samples)
print("number of samples / dataset length:", len(dataset))
print("item 1 / item 1:", dataset[1])

In [ ]:
# Exercise 3
# 
# values = [10, 11, 12, 13, 14], window_size = 2
# - x=[10, 11], y=12
# - x=[11, 12], y=13
# - x=[12, 13], y=14

class WindowDataset:
    def __init__(self, values, window_size):
        self.values = list(values)
        self.window_size = window_size

    def __len__(self):
        # TODO
        pass

    def __getitem__(self, index):
        # TODO
        pass


# ds = WindowDataset([10, 11, 12, 13, 14], window_size=2)
# print(len(ds))
# print(ds[0])

In [ ]:
# Exercise 3 Reference Solution

class WindowDatasetSolution:
    def __init__(self, values, window_size):
        self.values = list(values)
        self.window_size = window_size

    def __len__(self):
        return len(self.values) - self.window_size

    def __getitem__(self, index):
        window = self.values[index : index + self.window_size]
        target = self.values[index + self.window_size]
        return {"x": window, "y": target}


ds = WindowDatasetSolution([10, 11, 12, 13, 14], window_size=2)
print("length / length:", len(ds))
print(ds[0])
print(ds[1])
print(ds[2])

## Context Managers, Assertions, and Exceptions

These are not side topics. They are essential for writing reliable training code.

- safely manage resources
- perform quick sanity checks
- raise clear errors deliberately

In [ ]:
from io import StringIO


with StringIO("epoch=1,loss=0.82\nepoch=2,loss=0.64\n") as f:
    lines = [line.strip() for line in f if line.strip()]

print("log lines / log lines:", lines)


def validate_batch(features, labels):
    assert len(features) == len(labels), "features and labels must have the same length"
    if len(features) == 0:
        raise ValueError("batch must not be empty / batch must not be empty")
    return True


print("validation result / validation result:", validate_batch([[1, 2], [3, 4]], [0, 1]))

In [ ]:
# Exercise 4
# Debugging task
# The class below has at least two bugs.

class BrokenDataset:
    def __init__(self, rows):
        self.rows = rows

    def len(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.row[index]


# TODO:
# create an instance and verify len(ds) and ds[0]

In [ ]:
# Exercise 4 Reference Solution

class FixedDataset:
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.rows[index]


fixed_ds = FixedDataset([("a", 0), ("b", 1), ("c", 0)])
print("length / length:", len(fixed_ds))
print("first sample / first sample:", fixed_ds[0])

## Integrated Mini Exercise

This exercise combines the ideas above:

- classes
- `__len__` and `__getitem__`
- argument design
- input validation

In [ ]:
# Exercise 5
# 
# Requirements:
# returns number of samples
# returns (features, target)
# features follow feature_keys order
# raise KeyError if target_key is missing

records = [
    {"hours": 1.5, "attendance": 0.70, "passed": 0},
    {"hours": 3.0, "attendance": 0.90, "passed": 1},
    {"hours": 2.2, "attendance": 0.80, "passed": 1},
]


class MiniTabularDataset:
    def __init__(self, records, feature_keys, target_key):
        self.records = list(records)
        self.feature_keys = list(feature_keys)
        self.target_key = target_key

    def __len__(self):
        # TODO
        pass

    def __getitem__(self, index):
        # TODO
        pass


# ds = MiniTabularDataset(records, feature_keys=["hours", "attendance"], target_key="passed")
# print(len(ds))
# print(ds[0])

In [ ]:
# Exercise 5 Reference Solution

class MiniTabularDatasetSolution:
    def __init__(self, records, feature_keys, target_key):
        self.records = list(records)
        self.feature_keys = list(feature_keys)
        self.target_key = target_key

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        if self.target_key not in row:
            raise KeyError(f"missing target key: {self.target_key}")

        features = [row[key] for key in self.feature_keys]
        target = row[self.target_key]
        return features, target


ds = MiniTabularDatasetSolution(records, feature_keys=["hours", "attendance"], target_key="passed")
print("length / length:", len(ds))
print(ds[0])
print(ds[1])

## Summary

The most important outcome of this notebook is building the following mappings:

- training utilities
- classes and inheritance -> `nn.Module`, `Dataset`
- dataset interface
- more robust training code

You should now be able to answer:

1. Why do datasets usually implement `__len__` and `__getitem__`?
2. Why should lists usually not be used directly as default arguments?
3. What does `super()` actually do during subclass initialization?
4. Why do training scripts often start with sanity checks?

Suggested next step:

- Move to the NumPy core notebook and build strong `shape` intuition.